# prestructured

> Pre-structured (setpoint) policy with bump jumps (paper SI Eqs. 20-25).

In [ ]:
#| default_exp prestructured

In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
import math
from dataclasses import dataclass
import numpy as np
from explRL.agents import Agent, Policy, RL


In [ ]:
#| export
def wrap(a):
    "Heading in [0, 2*pi) as a float."
    return float(a) % (2 * np.pi)

def angdiff(a, b):
    "a - b wrapped into [-pi, pi); works element-wise on arrays."
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return (a - b + np.pi) % (2 * np.pi) - np.pi


Policy curves (Eqs. 20-21). `thetaC` is the compass heading, `thetaG` the goal heading; all accept arrays.

In [ ]:
#| export
def turnProb(thetaC, thetaG, p):
    "Eq. 20: probability of a clockwise saccade."
    return -p.GS / 2 * np.sin(angdiff(thetaC, thetaG)) + p.BS + 0.5

def driftRate(thetaC, thetaG, p):
    "Eq. 20: fixation drift rate; mean fixation duration is aF / driftRate."
    return p.GF / 2 * (1 - np.cos(angdiff(thetaC, thetaG))) + p.BF

def jumpProb(thetaC, thetaG, p):
    "Eq. 21 with the sign decided in spec Q1 (1 - cos): fewest jumps at the goal."
    return p.GJ / 2 * (1 - np.cos(angdiff(thetaC, thetaG))) + p.BJ


In [ ]:
#| export
def effectivePolicy(thetaA, thetaG, p):
    "Eq. 23: drift rate, turn probability, fixation duration and turn bias seen at arena headings thetaA."
    thetaA = np.asarray(thetaA, dtype=float)
    pj = jumpProb(thetaA, thetaG, p)
    nu = (1 - pj) * driftRate(thetaA, thetaG, p) + pj * driftRate(thetaA + p.thetaJ, thetaG, p)
    dS = (1 - pj) * turnProb(thetaA, thetaG, p) + pj * turnProb(thetaA + p.thetaJ, thetaG, p)
    return dict(nu=nu, dS=dS, fixDur=p.aF / nu, turnBias=2 * dS - 1)


In [ ]:
#| export
class prestructuredPolicy(Policy):
    "Policy whose only learnable state is the goal heading thetaG (compass coordinates, radians)."
    def __init__(self, thetaG):
        super().__init__()
        self.thetaG = wrap(thetaG)

@dataclass
class Event:
    "One fixation followed by one saccade and a possible bump jump."
    thetaA: float       # arena heading before the saccade
    thetaC: float       # compass heading before the saccade
    fixDur: float       # fixation duration, s
    direction: int      # +1 CW, -1 CCW
    dx: float           # saccade size with sign, radians
    jumped: bool        # bump jumped after the saccade
    thetaC_next: float  # compass heading after saccade and jump


In [ ]:
#| export
class prestructuredAgent(Agent):
    "Agent with an arena heading `theta` and a compass heading `thetaC` that can jump by `thetaJ`."
    def __init__(self, theta, goal, policy, rl, params, rng=None, thetaC=None):
        if not isinstance(policy, prestructuredPolicy):
            raise TypeError("policy must be a prestructuredPolicy instance")
        super().__init__(wrap(theta), goal, policy, rl, params)
        self.rng = rng if rng is not None else np.random.default_rng()
        self.thetaC = wrap(theta if thetaC is None else thetaC)

    def samplePolicy(self) -> Event:
        "Draw one event; the draw order (wald, direction, size, jump) is fixed for reproducibility."
        p, g, c, rng = self.params, self.policy.thetaG, self.thetaC, self.rng
        nu = float(driftRate(c, g, p))
        fixDur = float(rng.wald(p.aF / nu, p.aF ** 2 / p.sigF ** 2))
        direction = 1 if rng.random() < turnProb(c, g, p) else -1
        dx = direction * float(np.deg2rad(rng.lognormal(p.muS, p.sigmaS)))
        c1 = wrap(c + dx)
        jumped = bool(rng.random() < jumpProb(c1, g, p))
        return Event(self.theta, c, fixDur, direction, dx, jumped, wrap(c1 + p.thetaJ * jumped))


In [ ]:
#| export
class GoalHeadingRL(RL):
    "Policy-gradient update of the goal heading after each saccade (Eqs. 24-25, saccade term only)."
    def __init__(self, policy, lr=1.0, eps=1e-12):
        super().__init__(policy)
        self.lr, self.eps = float(lr), float(eps)

    def chooseAction(self, agent):
        "Sample the next event from the agent's policy."
        return agent.samplePolicy()

    def _clampedDS(self, thetaC, thetaG, p):
        return min(1 - self.eps, max(self.eps, float(turnProb(thetaC, thetaG, p))))

    def logPiS(self, agent, ev, thetaG=None) -> float:
        "Log-probability of the event's saccade direction under goal thetaG (default: current goal)."
        g = self.policy.thetaG if thetaG is None else float(thetaG)
        dS = self._clampedDS(ev.thetaC, g, agent.params)
        return math.log(dS) if ev.direction > 0 else math.log(1 - dS)

    def gradLogPiS(self, agent, ev) -> float:
        "d logPiS / d thetaG; uses +GS/2 cos (spec Q2; the printed Eq. 25 has the opposite sign)."
        p, g = agent.params, self.policy.thetaG
        dS = self._clampedDS(ev.thetaC, g, p)
        ddS = p.GS / 2 * math.cos(angdiff(ev.thetaC, g))
        return ddS / dS if ev.direction > 0 else -ddS / (1 - dS)

    def updateParams(self, agent, ev, reward):
        "thetaG <- wrap(thetaG + lr * reward * gradLogPiS)."
        self.policy.thetaG = wrap(self.policy.thetaG + self.lr * float(reward) * self.gradLogPiS(agent, ev))


Policy curves stay in valid ranges and respect heading asymmetry:

In [ ]:
from math import pi
from fastcore.test import test_eq, test_fail, test_close
from explRL.params import Params

p = Params()
g = pi
ths = np.linspace(0, 2 * pi, 37, endpoint=False)
dS = turnProb(ths, g, p)
nu = driftRate(ths, g, p)
pj = jumpProb(ths, g, p)
assert dS.min() >= 0.05 and dS.max() <= 0.95
assert nu.min() >= 0.05 and nu.max() <= 0.85 + 1e-9
assert pj.min() >= 0.05 and pj.max() <= 0.75
for i, th in enumerate(ths):
    test_close(turnProb(th, g, p), dS[i]); test_close(jumpProb(th, g, p), pj[i])
assert turnProb(g + 0.01, g, p) < 0.5 < turnProb(g - 0.01, g, p)


**AC-2**: Eq. 23 readout is bimodal; turn bias points toward or away from the goal:

In [ ]:
def theta_at_rel_deg(rel_deg, goal=pi):
    return wrap(goal + np.deg2rad(rel_deg))

eff = effectivePolicy(np.array([theta_at_rel_deg(r) for r in (0, 90, -90, 180)]), g, p)
fd0, fd90, fd180 = eff['fixDur'][0], eff['fixDur'][1], eff['fixDur'][3]
assert fd0 > fd90 and fd180 > fd90
for rel in (30, -30, 80, -80):
    tb = effectivePolicy(theta_at_rel_deg(rel), g, p)['turnBias']
    assert np.sign(tb) == -np.sign(rel)
for rel in (150, -150, 170, -170):
    tb = effectivePolicy(theta_at_rel_deg(rel), g, p)['turnBias']
    assert np.sign(tb) == np.sign(rel)


**AC-3**: With no jumps, effective readout matches Eq. 20 at the arena heading:

In [ ]:
p0 = Params(GJ=0.0, BJ=0.0)
th = np.linspace(0, 2 * pi, 25, endpoint=False)
eff0 = effectivePolicy(th, g, p0)
test_close(eff0['dS'], turnProb(th, g, p0))
test_close(eff0['nu'], driftRate(th, g, p0))
test_close(eff0['fixDur'], p0.aF / eff0['nu'])
assert eff0['fixDur'].argmax() == int(np.argmin(np.abs(angdiff(th, g))))


Agent sampling, wrapping, and draw order:

In [ ]:
def toy_prestructured(theta=-0.1, seed=0, thetaC=None, **kw):
    pol = prestructuredPolicy(pi)
    params = Params(**kw)
    return prestructuredAgent(theta, pi, pol, GoalHeadingRL(pol, lr=0.0), params,
                              rng=np.random.default_rng(seed), thetaC=thetaC)

a = toy_prestructured()
test_close(a.theta, 2 * pi - 0.1)
test_close(a.thetaC, a.theta)
test_fail(lambda: prestructuredAgent(0.0, pi, Policy(), GoalHeadingRL(prestructuredPolicy(pi)), Params()),
          contains='prestructuredPolicy')
ev = toy_prestructured(seed=4).samplePolicy()
assert ev.fixDur > 0
assert abs(ev.dx) > 0 and abs(ev.dx) != round(abs(ev.dx))
off = abs(angdiff(ev.thetaC_next, wrap(ev.thetaC + ev.dx)))
test_close(off, 0.0 if not ev.jumped else pi)
a0, a1 = toy_prestructured(seed=5), toy_prestructured(seed=5)
for _ in range(20):
    test_eq(a0.samplePolicy(), a1.samplePolicy())


Mean fixation duration at the anti-goal compass heading (heavy-tailed at the goal):

In [ ]:
ag = toy_prestructured(thetaC=0.0, seed=99)
ag.policy.thetaG = pi
nu_ag = float(driftRate(ag.thetaC, ag.policy.thetaG, ag.params))
pred = ag.params.aF / nu_ag
durs = [ag.samplePolicy().fixDur for _ in range(20000)]
test_close(np.mean(durs), pred, eps=0.05 * pred)


**AC-6**: Observed CW and jump rates match predictions (20 000 calls):

In [ ]:
ag = toy_prestructured(thetaC=1.0, seed=7)
ag.policy.thetaG = 2.0
dS_pred = float(turnProb(ag.thetaC, ag.policy.thetaG, ag.params))
cw, jumped, post_c = [], [], []
for _ in range(20000):
    ev = ag.samplePolicy()
    cw.append(ev.direction > 0)
    jumped.append(ev.jumped)
    post_c.append(wrap(ag.thetaC + ev.dx))
test_close(np.mean(cw), dS_pred, eps=0.01)
pj_pred = float(np.mean([jumpProb(c1, ag.policy.thetaG, ag.params) for c1 in post_c]))
test_close(np.mean(jumped), pj_pred, eps=0.01)


**AC-4**: Analytic goal gradient matches finite differences of `logPiS`:

In [ ]:
def fd_logPiS(rl, agent, ev, h=1e-6):
    g0 = rl.policy.thetaG
    g = lambda x: rl.logPiS(agent, ev, thetaG=x)
    return (g(g0 + h) - g(g0 - h)) / (2 * h)

rl = GoalHeadingRL(prestructuredPolicy(0.0))
agent = toy_prestructured(thetaC=0.5, seed=1)
agent.policy = rl.policy
agent.rl = rl
for d in (1, -1):
    for tc in (0.2, 1.1, 5.0):
        ev = Event(agent.theta, tc, 0.1, d, 0.05 * d, False, wrap(tc + 0.05 * d))
        test_close(rl.gradLogPiS(agent, ev), fd_logPiS(rl, agent, ev), eps=1e-6)


**AC-5**: Small learning steps move `dS` in the reward direction:

In [ ]:
def dS_at(agent, tc=None):
    tc = agent.thetaC if tc is None else tc
    return float(turnProb(tc, agent.policy.thetaG, agent.params))

rl = GoalHeadingRL(prestructuredPolicy(pi), lr=0.01)
agent = toy_prestructured(seed=2)
agent.policy = rl.policy
agent.rl = rl
ev = Event(agent.theta, agent.thetaC, 0.1, 1, 0.02, False, wrap(agent.thetaC + 0.02))
before = dS_at(agent)
rl.updateParams(agent, ev, 1.0)
assert dS_at(agent) > before
rl.policy.thetaG = pi
before = dS_at(agent)
rl.updateParams(agent, ev, -1.0)
assert dS_at(agent) < before
g_fixed = rl.policy.thetaG
rl.lr = 0.0
rl.updateParams(agent, ev, 1.0)
test_close(rl.policy.thetaG, g_fixed)
assert np.isfinite(rl.policy.thetaG)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()